# BiAE: ShARC loader and tiny explicit-alignment demo

This notebook accompanies [Luo et al. (2023)](https://aclanthology.org/2023.findings-emnlp.866/). It loads the real public ShARC dataset and illustrates explicit document-to-user alignment without training the full BiAE system.

In [ ]:
import io
import json
import re
import urllib.request
import zipfile
from collections import Counter
from pprint import pprint

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Download and load ShARC

The official archive is distributed under CC BY-SA 3.0. We locate the development JSON by filename so the cell is not sensitive to the archive's top-level directory name.

In [ ]:
SHARC_URL = 'https://sharc-data.github.io/data/sharc1-official.zip'
with urllib.request.urlopen(SHARC_URL) as response:
    archive_bytes = response.read()

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    dev_name = next(name for name in archive.namelist() if name.endswith('/json/sharc_dev.json'))
    sharc_dev = json.loads(archive.read(dev_name))

print(f'Loaded {len(sharc_dev):,} ShARC development examples from {dev_name}.')
for row in sharc_dev[:2]:
    pprint({key: row[key] for key in ('utterance_id', 'snippet', 'question', 'scenario', 'history', 'answer')})
    print()

## 2. Inspect decision labels

ShARC answers `Yes`, `No`, or `Irrelevant` directly. Any other answer is a generated follow-up question and corresponds to the paper's `MORE` action.

In [ ]:
def decision_label(answer):
    normalized = answer.strip().lower()
    return normalized.upper() if normalized in {'yes', 'no', 'irrelevant'} else 'MORE'

label_counts = Counter(decision_label(row['answer']) for row in sharc_dev)
pd.Series(label_counts, name='count').sort_values(ascending=False).to_frame()

## 3. Tiny explicit-alignment demonstration

BiAE learns an alignment distribution between document hypotheses and user premises. For a fast, dependency-light illustration, this cell splits one real ShARC item and uses TF-IDF cosine similarity. Higher values indicate stronger lexical alignment.

In [ ]:
example = next(row for row in sharc_dev if row['scenario'].strip() and row['history'])

def split_document(text):
    parts = re.split(r'\n+|(?<=[.!?])\s+', text)
    return [part.strip(' -*•\t') for part in parts if part.strip(' -*•\t')]

document_hypotheses = split_document(example['snippet'])
user_premises = [example['scenario'].strip()] + [
    f"{turn['follow_up_question']} {turn['follow_up_answer']}"
    for turn in example['history']
]

all_text = document_hypotheses + user_premises
vectors = TfidfVectorizer(stop_words='english').fit_transform(all_text)
m = len(document_hypotheses)
alignment = cosine_similarity(vectors[:m], vectors[m:])

print('Question:', example['question'])
print('Gold answer:', example['answer'])
display(pd.DataFrame(
    alignment,
    index=[f'D{i}: {text[:70]}' for i, text in enumerate(document_hypotheses, 1)],
    columns=[f'U{i}: {text[:55]}' for i, text in enumerate(user_premises, 1)],
).style.background_gradient(cmap='Blues', axis=None).format('{:.2f}'))

### Scope

This notebook verifies that the dataset remains directly usable and makes the alignment concept visible. It does not reproduce the paper's reported accuracy: the linked repository currently lacks the BiAE source code, and the full experiment requires learned DeBERTaV3/Sentence-BERT alignment, many-to-many entailment training, and T5 question generation.